In [ ]:
import math, torch, numpy as np, gpytorch, pandas as pd
from pathlib import Path
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle
import warnings; warnings.filterwarnings('ignore')

GLOBAL_SEED = 4203
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

from gpytorch.models import ExactGP
from gpytorch.likelihoods import DirichletClassificationLikelihood
from gpytorch.means import ConstantMean
from gpytorch.kernels import ScaleKernel, MaternKernel
from gpytorch.constraints import Interval

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Load ground truth ──────────────────────────────────────────────────────
GT_CSV = Path('data/ground_truth_phase_fractions.csv')
if not GT_CSV.exists():
    raise FileNotFoundError('Run RealData_GroundTruth_From_PhaseFractions.ipynb first.')

gt_df = pd.read_csv(GT_CSV)
if 'dominant_phase' not in gt_df.columns:
    cols = ['monoclinic_phase_fraction', 'orthorhombic_phase_fraction', 'tetragonal_phase_fraction']
    gt_df['dominant_phase'] = gt_df[cols].idxmax(axis=1).str.replace('_phase_fraction', '', regex=False)
    gt_df['dominant_phase_fraction'] = gt_df[cols].max(axis=1)

# ── Grid definition ────────────────────────────────────────────────────────
# Voltage : 506 → 716 V,  step 42 V  → 6 values
# Duration: 0.1 → 10.1 ms, step 2.5 ms → 5 values   (2.6 ms in grid ↔ 2.5 ms in GT, tol=0.15)
# Composition: linspace(0,1,16)  →  ZrO2 fraction
# Discrete space: 6 × 5 × 16 = 480 points

VOLTAGES     = [506, 548, 590, 632, 674, 716]
DURATIONS    = [0.1, 2.6, 5.1, 7.6, 10.1]
N_COMP       = 16
COMPOSITIONS = np.linspace(0.0, 1.0, N_COMP)
VD_GRID      = [(v, d) for v in VOLTAGES for d in DURATIONS]   # 30 conditions
N_VD         = len(VD_GRID)

V_MIN, V_MAX = 506.0, 716.0
D_MIN, D_MAX = 0.1, 10.1
C_STEP       = 1.0 / (N_COMP - 1)

norm_v   = lambda v: (v - V_MIN) / (V_MAX - V_MIN)
norm_d   = lambda d: (d - D_MIN) / (D_MAX - D_MIN)
denorm_v = lambda v: v * (V_MAX - V_MIN) + V_MIN   # works for scalars AND arrays
denorm_d = lambda d: d * (D_MAX - D_MIN) + D_MIN

# ── Ground-truth lookups ──────────────────────────────────────────────────
PHASE_TO_LABEL = {'monoclinic': 0, 'orthorhombic': 1, 'tetragonal': 2}
LABEL_NAME     = {0: 'Monoclinic', 1: 'Orthorhombic', 2: 'Tetragonal'}
CRYSTALLINE_THRESHOLD = 0.10
DURATION_TOL          = 0.15    # handles 2.6 ms grid → 2.5 ms GT

def _cond(v, d):
    return gt_df[(gt_df['voltage_V'] == v) & (abs(gt_df['duration_ms'] - d) <= DURATION_TOL)]

def _near(df, c):
    return df.loc[(df['zro2_fraction'] - c).abs().idxmin()]

def lookup_2C(v, d, c):
    rows = _cond(v, d)
    if rows.empty: return 0
    r = _near(rows, c)
    psum = r['monoclinic_phase_fraction'] + r['orthorhombic_phase_fraction'] + r['tetragonal_phase_fraction']
    return 1 if psum > CRYSTALLINE_THRESHOLD else 0

def lookup_3C(v, d, c):
    """Returns 0/1/2 (mono/ortho/tetra) or -1 if amorphous/outside."""
    rows = _cond(v, d)
    if rows.empty: return -1
    r = _near(rows, c)
    psum = r['monoclinic_phase_fraction'] + r['orthorhombic_phase_fraction'] + r['tetragonal_phase_fraction']
    if psum <= CRYSTALLINE_THRESHOLD: return -1
    return PHASE_TO_LABEL.get(r['dominant_phase'], -1)

def snap_vd(v_n, d_n):
    v = float(denorm_v(float(v_n))); d = float(denorm_d(float(d_n)))
    return (min(VOLTAGES, key=lambda x: abs(x-v)),
            min(DURATIONS, key=lambda x: abs(x-d)))

print(f'GT loaded: {len(gt_df)} rows')
print(f'Sanity — (716V,10.1ms,c=0.5): 2C={lookup_2C(716,10.1,0.5)}, 3C={lookup_3C(716,10.1,0.5)}')
print(f'Sanity — (506V, 0.1ms,c=0.5): 2C={lookup_2C(506, 0.1,0.5)}, 3C={lookup_3C(506, 0.1,0.5)}')

In [ ]:
# ── GP Model ───────────────────────────────────────────────────────────────
class DirichletGPModel(ExactGP):
    def __init__(self, train_x, train_y, likelihood, num_classes, nu,
                 ls_constraint=None, os_constraint=None):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = ConstantMean(batch_shape=torch.Size((num_classes,)))
        self.covar_module = ScaleKernel(
            MaternKernel(
                ard_num_dims=train_x.shape[1],
                batch_shape=torch.Size((num_classes,)),
                nu=nu, lengthscale_constraint=ls_constraint,
            ),
            batch_shape=torch.Size((num_classes,)),
            outputscale_constraint=os_constraint,
        )
    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(
            self.mean_module(x), self.covar_module(x))


def run_sweeps(train_x, train_y, n_iter, nu=2.5,
               init_ls=None, ls_con=None, os_con=None, lr=0.05, verbose=True):
    lik = DirichletClassificationLikelihood(train_y, learn_additional_noise=True)
    mdl = DirichletGPModel(train_x, lik.transformed_targets, lik,
                           num_classes=lik.num_classes, nu=nu,
                           ls_constraint=ls_con, os_constraint=os_con)
    if init_ls is not None:
        try:
            mdl.covar_module.base_kernel.lengthscale = init_ls
        except Exception:
            pass   # shape mismatch when num_classes changes — start fresh

    mdl.train(); lik.train()
    opt = torch.optim.Adam(mdl.parameters(), lr=lr)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(lik, mdl)

    for i in range(n_iter):
        opt.zero_grad()
        loss = -mll(mdl(train_x), lik.transformed_targets).mean()
        loss.backward(); opt.step()
        if verbose and i % 25 == 0:
            ls = mdl.covar_module.base_kernel.lengthscale.mean().item()
            print(f'  [{i+1:3d}/{n_iter}] loss={loss.item():.3f}  mean_ls={ls:.3f}')

    mdl.eval(); lik.eval()
    return mdl, lik


def acquisition(model, test_x, func='entropy', var_space=None, num_samples=256):
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        f_s = model(test_x).sample(torch.Size([num_samples]))
        p_s = torch.softmax(f_s, dim=1)
        probs = p_s.mean(dim=0)
        var   = p_s.var(dim=0).sum(dim=0)
        if func == 'entropy':
            unc = -torch.sum(probs * torch.log(probs + 1e-10), dim=0)
        elif func == 'most_unprobable':
            unc = 1 - probs.max(dim=0).values
        elif func == 'variance':
            unc = (p_s.var(dim=0, correction=0).sum(dim=0)
                   if var_space == 'prob'
                   else f_s.var(dim=0, correction=0).sum(dim=0))
        else:
            raise ValueError(func)
        entropy = float((-torch.sum(probs * torch.log(probs + 1e-10), dim=0)).mean())
    return probs, var, unc, entropy


def predict_vis(model, test_x, num_samples=128):
    """Returns probs [C,N], entropy [N] for visualization."""
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        f_s = model(test_x).sample(torch.Size([num_samples]))
        p_s = torch.softmax(f_s, dim=1)
        probs = p_s.mean(dim=0)
        unc   = -torch.sum(probs * torch.log(probs + 1e-10), dim=0)
    return probs, unc


def select_next(unc, exclude):
    """argmax uncertainty, masking out already-sampled indices."""
    u = unc.clone()
    for i in exclude:
        if i < len(u): u[i] = -float('inf')
    return int(u.argmax().item())


def acquisition_4class(model_2c, model_3c, test_x, lambda_2c=0.0, num_samples=256,
                       score_mode='phase_weighted', p_cryst_power=2.0):
    """3C-stage acquisition using the current 2C and 3C posteriors.

    score_mode options:
      - '4class_entropy': H([P(amorph), P(cryst)*P(phase|cryst)])
      - 'phase_weighted': P(cryst)^p * H_3c + lambda_2c * H_2c
      - 'phase_margin':   P(cryst)^p * (1 - max P_3c) + lambda_2c * H_2c
      - 'phase_variance': P(cryst)^p * sum Var[P_3c samples] + lambda_2c * H_2c
      - 'phase_only':     H_3c without 2C weighting

    ent_conv = mean(P_cryst * H_3c) over test_x, a fixed-reference convergence metric.
    """
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        # 2C posteriors
        f2_s    = model_2c(test_x).sample(torch.Size([num_samples]))
        p2      = torch.softmax(f2_s, dim=1).mean(dim=0)    # [2, N]
        p_cryst = p2[1]                                    # [N]
        H_2c    = -(p2 * torch.log(p2 + 1e-10)).sum(dim=0)  # [N]

        # 3C posteriors
        f3_s = model_3c(test_x).sample(torch.Size([num_samples]))
        p3_s = torch.softmax(f3_s, dim=1)                   # [S, 3, N]
        p3   = p3_s.mean(dim=0)                             # [3, N]
        H_3c = -(p3 * torch.log(p3 + 1e-10)).sum(dim=0)     # [N]
        margin_3c = 1.0 - p3.max(dim=0).values              # [N]
        var_3c = p3_s.var(dim=0, correction=0).sum(dim=0)   # [N]

        # Effective 4-class distribution
        p_eff = torch.stack([
            1.0 - p_cryst,       # amorphous
            p_cryst * p3[0],     # mono
            p_cryst * p3[1],     # ortho
            p_cryst * p3[2],     # tetra
        ], dim=0)                                             # [4, N]
        H_4c = -(p_eff * torch.log(p_eff + 1e-10)).sum(dim=0)  # [N]
        p_weight = p_cryst.clamp(0.0, 1.0) ** p_cryst_power

        if score_mode == '4class_entropy':
            scores = H_4c + lambda_2c * H_2c
        elif score_mode == 'phase_weighted':
            scores = p_weight * H_3c + lambda_2c * H_2c
        elif score_mode == 'phase_margin':
            scores = p_weight * margin_3c + lambda_2c * H_2c
        elif score_mode == 'phase_variance':
            scores = p_weight * var_3c + lambda_2c * H_2c
        elif score_mode == 'phase_only':
            scores = H_3c
        else:
            raise ValueError(f'Unknown score_mode: {score_mode}')

        ent_conv = float((p_cryst * H_3c).mean().item())

    return scores, ent_conv


In [ ]:
# Discrete acquisition grid: 480 points
test_x_all = torch.tensor(
    [[norm_v(v), norm_d(d), c] for v, d in VD_GRID for c in COMPOSITIONS],
    dtype=torch.float32)   # [480, 3]

gt_y_2c = torch.tensor(
    [lookup_2C(v, d, c) for v, d in VD_GRID for c in COMPOSITIONS],
    dtype=torch.long)

gt_y_3c = torch.tensor(
    [lookup_3C(v, d, c) for v, d in VD_GRID for c in COMPOSITIONS],
    dtype=torch.long)   # -1=amorphous (or no GT row), 0=mono, 1=ortho, 2=tetra

# Accuracy tracking against full ground truth grid
ACCURACY_OUT_DIR = Path('outputs/accuracy')
ACCURACY_CSV = ACCURACY_OUT_DIR / '2c3c_accuracy_history.csv'
ACCURACY_PNG = ACCURACY_OUT_DIR / '2c3c_accuracy_history.png'
ACCURACY_SAMPLES = 256

accuracy_history = []

def _predict_class_labels(model, x, num_samples=ACCURACY_SAMPLES):
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        f_s = model(x).sample(torch.Size([num_samples]))
        probs = torch.softmax(f_s, dim=1).mean(dim=0)
    return probs.argmax(dim=0), probs


def compute_2c_accuracy(model_2c, probs_2c=None):
    if probs_2c is None:
        pred, _ = _predict_class_labels(model_2c, test_x_all)
    else:
        pred = probs_2c.argmax(dim=0)
    correct = (pred.cpu() == gt_y_2c.cpu())
    return float(correct.float().mean().item()), int(correct.numel())



def compute_3c_accuracy(model_3c):
    # 3C model is conditional on crystalline points, so evaluate only where GT is crystalline.
    mask = gt_y_3c >= 0
    if int(mask.sum().item()) == 0:
        return np.nan, 0, np.nan
    pred, _ = _predict_class_labels(model_3c, test_x_all)
    pred_eval = pred.cpu()[mask.cpu()]
    gt_eval = gt_y_3c.cpu()[mask.cpu()]
    correct = pred_eval == gt_eval

    class_acc = []
    for lbl in [0, 1, 2]:
        class_mask = gt_eval == lbl
        if class_mask.any():
            class_acc.append(float((pred_eval[class_mask] == lbl).float().mean().item()))
    bal_acc = float(np.mean(class_acc)) if class_acc else np.nan
    return float(correct.float().mean().item()), int(correct.numel()), bal_acc
def accuracy_history_df():
    cols = [
        'iteration', 'stage', 'stage_iteration',
        'accuracy_2c', 'accuracy_3c', 'balanced_accuracy_3c', 'n_eval_2c', 'n_eval_3c'
    ]
    return pd.DataFrame(accuracy_history, columns=cols)


def plot_accuracy_history(df=None, save_path=ACCURACY_PNG, show=False):
    if df is None:
        df = accuracy_history_df()
    if df.empty:
        return None

    fig, ax = plt.subplots(figsize=(8.5, 4.8), facecolor='white')
    ax.plot(df['iteration'], df['accuracy_2c'], marker='o', linewidth=2.0,
            color='#C0392B', label='2C crystallized accuracy')

    has_3c = df['accuracy_3c'].notna()
    if has_3c.any():
        ax.plot(df.loc[has_3c, 'iteration'], df.loc[has_3c, 'accuracy_3c'],
                marker='s', linewidth=2.0, color='#1F77B4',
                label='3C phase accuracy')

    ax.set_xlabel('Iteration')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0.0, 1.02)
    x_min = float(df['iteration'].min())
    x_max = float(df['iteration'].max())
    ax.set_xlim(x_min - 0.5, x_max + 0.5) if x_min == x_max else ax.set_xlim(x_min, x_max)
    ax.grid(True, alpha=0.30)
    ax.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='#cccccc')
    ax.set_title('Accuracy vs Iteration')
    fig.tight_layout()

    ACCURACY_OUT_DIR.mkdir(exist_ok=True)
    fig.savefig(save_path, dpi=200, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


def save_accuracy_outputs(show=False):
    ACCURACY_OUT_DIR.mkdir(exist_ok=True)
    df = accuracy_history_df()
    df.to_csv(ACCURACY_CSV, index=False)
    plot_accuracy_history(df, ACCURACY_PNG, show=show)
    return df


def load_accuracy_history(csv_path=ACCURACY_CSV):
    return pd.read_csv(csv_path)


def replay_accuracy_plot(csv_path=ACCURACY_CSV, show=True):
    df = load_accuracy_history(csv_path)
    return plot_accuracy_history(df, save_path=ACCURACY_PNG, show=show)


def record_accuracy(stage, stage_iteration, model_2c, model_3c=None, probs_2c=None, show=False):
    acc_2c, n_2c = compute_2c_accuracy(model_2c, probs_2c=probs_2c)
    if model_3c is None:
        acc_3c, bal_acc_3c, n_3c = np.nan, np.nan, 0
    else:
        acc_3c, n_3c, bal_acc_3c = compute_3c_accuracy(model_3c)

    row = {
        'iteration': len(accuracy_history) + 1,
        'stage': stage,
        'stage_iteration': int(stage_iteration),
        'accuracy_2c': acc_2c,
        'accuracy_3c': acc_3c,
        'balanced_accuracy_3c': bal_acc_3c,
        'n_eval_2c': n_2c,
        'n_eval_3c': n_3c,
    }
    accuracy_history.append(row)
    save_accuracy_outputs(show=show)

    msg = f'  Accuracy: 2C={acc_2c:.3f} (n={n_2c})'
    if model_3c is not None:
        msg += f' | 3C={acc_3c:.3f}, balanced={bal_acc_3c:.3f} (n={n_3c}, GT crystalline only)'
    print(msg)
    return row

# Fine visualization grids
V_fine = np.linspace(506, 716, 70)
D_fine = np.linspace(0.1, 10.1, 50)
C_fine = np.linspace(0.0, 1.0, 70)
VV, DD = np.meshgrid(V_fine, D_fine)   # both (len(D_fine), len(V_fine))
SHP    = DD.shape
VIS_COMP = [0.2, 0.5, 0.8]

vis_grids = {
    c: torch.tensor(np.column_stack([
        (VV.ravel() - V_MIN) / (V_MAX - V_MIN),
        (DD.ravel() - D_MIN) / (D_MAX - D_MIN),
        np.full(VV.size, c)
    ]), dtype=torch.float32)
    for c in VIS_COMP
}   # each [len(V_fine) * len(D_fine), 3]

def _grid_edges(values, lower=None, upper=None):
    values = np.asarray(values, dtype=float)
    mids = (values[:-1] + values[1:]) / 2
    first = values[0] - (mids[0] - values[0])
    last = values[-1] + (values[-1] - mids[-1])
    edges = np.r_[first, mids, last]
    if lower is not None:
        edges[0] = lower
    if upper is not None:
        edges[-1] = upper
    return edges

V_EDGES = _grid_edges(VOLTAGES)
D_EDGES = _grid_edges(DURATIONS)
C_EDGES = _grid_edges(COMPOSITIONS, lower=0.0, upper=1.0)
VOX_V, VOX_D, VOX_C = np.meshgrid(V_EDGES, D_EDGES, C_EDGES, indexing='ij')

# Reshape to [V, D, C] arrays for display
gt_2c_grid = gt_y_2c.view(len(VOLTAGES), len(DURATIONS), N_COMP).numpy().astype(float)
gt_3c_grid = gt_y_3c.view(len(VOLTAGES), len(DURATIONS), N_COMP).numpy().astype(float)

from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import to_rgba, ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D

CRYST_GT_COL  = '#C0392B'
AMORPH_GT_COL = '#6C8EBF'

GT_CMAP_2C = ListedColormap([AMORPH_GT_COL, CRYST_GT_COL])
GT_NORM_2C = BoundaryNorm([-0.5, 0.5, 1.5], GT_CMAP_2C.N)

# 3C colormap: amorphous (grey-blue) + 3 phases matching PHASE_COLS in cell-viz
GT_CMAP_3C = ListedColormap([
    AMORPH_GT_COL,
    [0.706, 0.016, 0.149],   # Monoclinic:   crimson
    [0.016, 0.361, 0.733],   # Orthorhombic: steel blue
    [0.906, 0.671, 0.141],   # Tetragonal:   amber
])
GT_NORM_3C = BoundaryNorm([-1.5, -0.5, 0.5, 1.5, 2.5], GT_CMAP_3C.N)

# ── 2C Ground Truth ────────────────────────────────────────────────────────
fig, axs = plt.subplots(1, len(DURATIONS), figsize=(19, 3.8),
                        sharey=True, constrained_layout=True)
fig.suptitle('Ground Truth: 2C (Crystalline / Amorphous) by duration slice',
             fontsize=13, fontweight='bold')
for ax, d_val, di in zip(axs, DURATIONS, range(len(DURATIONS))):
    im = ax.pcolormesh(V_EDGES, C_EDGES, gt_2c_grid[:, di, :].T,
                       cmap=GT_CMAP_2C, norm=GT_NORM_2C, shading='flat')
    ax.set_title(f'{d_val:.1f} ms', fontsize=9)
    ax.set_xlabel('Voltage (V)')
    ax.set_xticks(VOLTAGES)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.grid(color='white', linewidth=0.7, alpha=0.55)
axs[0].set_ylabel('ZrO₂ fraction')
cbar = fig.colorbar(im, ax=axs.ravel().tolist(), ticks=[0, 1], shrink=0.88)
cbar.set_ticklabels(['Amorphous', 'Crystalline'])
plt.show()

# ── 3C Ground Truth ────────────────────────────────────────────────────────
fig, axs = plt.subplots(1, len(DURATIONS), figsize=(19, 3.8),
                        sharey=True, constrained_layout=True)
fig.suptitle('Ground Truth: 3C (Phase Identity) by duration slice',
             fontsize=13, fontweight='bold')
for ax, d_val, di in zip(axs, DURATIONS, range(len(DURATIONS))):
    im = ax.pcolormesh(V_EDGES, C_EDGES, gt_3c_grid[:, di, :].T,
                       cmap=GT_CMAP_3C, norm=GT_NORM_3C, shading='flat')
    ax.set_title(f'{d_val:.1f} ms', fontsize=9)
    ax.set_xlabel('Voltage (V)')
    ax.set_xticks(VOLTAGES)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.grid(color='white', linewidth=0.7, alpha=0.55)
axs[0].set_ylabel('ZrO₂ fraction')
cbar = fig.colorbar(im, ax=axs.ravel().tolist(),
                    ticks=[-1, 0, 1, 2], shrink=0.88)
cbar.set_ticklabels(['Amorphous', 'Monoclinic', 'Orthorhombic', 'Tetragonal'])
plt.show()

print(f'2C: Crystalline={int(gt_y_2c.sum())}/{len(gt_y_2c)}')
print(f'3C: Mono={(gt_y_3c==0).sum()},  Ortho={(gt_y_3c==1).sum()},  '
      f'Tetra={(gt_y_3c==2).sum()},  Amorph={(gt_y_3c==-1).sum()}')

In [ ]:
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import to_rgba, ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D

# ── Color palette ─────────────────────────────────────────────────────────
CRYST_COL  = '#C0392B'
AMORPH_COL = '#6C8EBF'

PHASE_COLS = {0: [0.706, 0.016, 0.149],
              1: [0.016, 0.361, 0.733],
              2: [0.906, 0.671, 0.141]}

_PC_RGB = np.array([PHASE_COLS[k] for k in sorted(PHASE_COLS)], dtype=float)

PROB_CMAP = 'Greys'
FIGURE_OUT_DIR = Path('outputs/figures')

# ── PPT style constants ────────────────────────────────────────────────────
_FL  = 15   # axis label font
_FT  = 12   # tick font
_FSL = 13   # slice/panel title font
_FSP = 13   # suptitle font
_SC  = 200  # circle scatter size
_SS  = 450  # star scatter size


def _save_iter_fig(fig, stage, iteration, suffix):
    out_dir = FIGURE_OUT_DIR / stage
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{stage}_iter_{iteration+1:03d}_{suffix}.png'
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    print(f'  Saved figure: {out_path}')
    return out_path


def _tnp(x):
    return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)


def _predict_np(model, x, samples=128):
    probs, unc = predict_vis(model, x, num_samples=samples)
    return _tnp(probs), _tnp(unc)


def _pcryst_grid(model, samples=128):
    probs, _ = _predict_np(model, test_x_all, samples=samples)
    return probs[1].reshape(len(VOLTAGES), len(DURATIONS), N_COMP)


def _make_vc_grid(d_fixed):
    vvc, cvc = np.meshgrid(V_fine, C_fine)
    x = np.column_stack([
        norm_v(vvc.ravel()),
        np.full(vvc.size, norm_d(d_fixed)),
        cvc.ravel(),
    ])
    return vvc, cvc, torch.tensor(x, dtype=torch.float32), cvc.shape


def _make_dc_grid(v_fixed):
    ddc, cdc = np.meshgrid(D_fine, C_fine)
    x = np.column_stack([
        np.full(ddc.size, norm_v(v_fixed)),
        norm_d(ddc.ravel()),
        cdc.ravel(),
    ])
    return ddc, cdc, torch.tensor(x, dtype=torch.float32), cdc.shape


def _style_3d(ax):
    ax.set_facecolor('#F8F8F6')
    ax.xaxis.pane.set_facecolor((0.96, 0.94, 0.90, 0.72))
    ax.yaxis.pane.set_facecolor((0.95, 0.97, 0.99, 0.78))
    ax.zaxis.pane.set_facecolor((0.98, 0.98, 0.98, 0.82))
    for _a in [ax.xaxis, ax.yaxis, ax.zaxis]:
        _a.pane.set_edgecolor((1, 1, 1, 0))
        _a._axinfo['grid']['color'] = (0.72, 0.72, 0.72, 0.42)
        _a._axinfo['grid']['linewidth'] = 0.8


def _setup_3d_axes(ax, aspect=(1.45, 0.90, 1.15)):
    ax.set_xlim(V_EDGES[0], V_EDGES[-1])
    ax.set_ylim(D_EDGES[0], D_EDGES[-1])
    ax.set_zlim(0.0, 1.0)
    ax.set_box_aspect(aspect)
    ax.set_xlabel('Voltage (V)',    labelpad=9,  fontsize=10, fontweight='bold')
    ax.set_ylabel('Duration (ms)', labelpad=9,  fontsize=10, fontweight='bold')
    ax.set_zlabel('ZrO2 fraction', labelpad=8,  fontsize=10, fontweight='bold')
    ax.set_xticks(VOLTAGES); ax.set_yticks(DURATIONS)
    ax.set_zticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.tick_params(labelsize=8)


def _safe_contour(ax, x, y, z, levels, **kwargs):
    z_arr = np.asarray(z, dtype=float)
    if not np.isfinite(z_arr).any():
        return None
    zmin, zmax = np.nanmin(z_arr), np.nanmax(z_arr)
    valid = [lv for lv in np.atleast_1d(levels) if zmin < lv < zmax]
    if not valid:
        return None
    return ax.contour(x, y, z_arr, levels=valid, **kwargs)


def _draw_pcryst_volume(ax, p_grid, threshold=0.5):
    filled = p_grid >= threshold
    if not filled.any():
        ax.text2D(0.03, 0.93, f'No voxels >= {threshold:.2f}', transform=ax.transAxes)
        return
    color = np.array(to_rgba(CRYST_COL, 1.0))
    fc = np.zeros(p_grid.shape + (4,), dtype=float)
    fc[..., :3] = color[:3]
    scaled = np.clip((p_grid - threshold) / (1 - threshold + 1e-10), 0, 1)
    fc[..., 3] = np.where(filled, 0.18 + 0.42 * scaled, 0.0)
    ax.voxels(VOX_V, VOX_D, VOX_C, filled, facecolors=fc,
              edgecolor=to_rgba('white', 0.28), linewidth=0.28, shade=False)


def _add_crystal_contours_3d(ax, model_2c, threshold=0.5, color='black'):
    for c_vis in VIS_COMP:
        probs, _ = _predict_np(model_2c, vis_grids[c_vis], samples=96)
        p = probs[1].reshape(SHP)
        if p.min() < threshold < p.max():
            ax.contour(VV, DD, p, levels=[threshold], zdir='z', offset=c_vis,
                       colors=color, linewidths=1.5, alpha=0.95)


def _scatter_train_3d(ax, train_x, train_y, mode='2c'):
    if len(train_x) == 0:
        return
    x_np = _tnp(train_x); y_np = _tnp(train_y).astype(int)
    v = denorm_v(x_np[:, 0]); d = denorm_d(x_np[:, 1]); c = x_np[:, 2]
    if mode == '2c':
        items = [(0, AMORPH_COL, 'Amorphous'), (1, CRYST_COL, 'Crystalline')]
    else:
        items = [(lbl, PHASE_COLS[lbl], LABEL_NAME[lbl]) for lbl in [0, 1, 2]]
    for lbl, col, lab in items:
        mask = y_np == lbl
        if mask.any():
            ax.scatter(v[mask], d[mask], c[mask], color=col,
                       s=120 if mode == '2c' else 135,
                       edgecolors='white', linewidths=0.9,
                       depthshade=False, zorder=10, label=lab)


def _scatter_next_3d(ax, v_n, d_n, c_n):
    ax.scatter(v_n, d_n, c_n, color='gold', s=330, marker='*',
               edgecolors='black', linewidths=1.1,
               depthshade=False, zorder=12, label='Next')


# ── 2C iteration plot ──────────────────────────────────────────────────────
def plot_2c_iter(model, train_x, train_y, next_idx, sampled, iteration):
    pt  = test_x_all[next_idx]
    v_n = float(denorm_v(pt[0]))
    d_n = float(denorm_d(pt[1]))
    c_n = float(pt[2])

    p_grid = _pcryst_grid(model, samples=128)

    # ── Figure 1: 3D voxel ────────────────────────────────────────────────
    fig3d = plt.figure(figsize=(11, 7), facecolor='white')
    ax3d  = fig3d.add_subplot(111, projection='3d')
    _style_3d(ax3d); _setup_3d_axes(ax3d)
    _draw_pcryst_volume(ax3d, p_grid, threshold=0.5)
    _add_crystal_contours_3d(ax3d, model, threshold=0.5, color='black')
    _scatter_train_3d(ax3d, train_x, train_y, mode='2c')
    _scatter_next_3d(ax3d, v_n, d_n, c_n)
    ax3d.set_title(
        f'2C Iteration {iteration+1}  |  Next = {v_n:.0f} V / {d_n:.1f} ms / ZrO2={c_n:.2f}\n'
        'red voxels: P(crystalline) >= 0.5   |   black contours: 0.5 boundary',
        fontsize=11, fontweight='bold', pad=12)
    ax3d.legend(loc='upper left', bbox_to_anchor=(-0.02, 0.98), fontsize=8,
                frameon=True, facecolor='white', edgecolor='#cccccc')
    ax3d.view_init(elev=23, azim=-55)
    plt.tight_layout()
    _save_iter_fig(fig3d, '2c', iteration, 'volume')
    plt.show()

    # ── Figure 2: duration slices (PPT style) ────────────────────────────
    x_np = _tnp(train_x); y_np = _tnp(train_y).astype(int)
    v_tr = denorm_v(x_np[:, 0]); d_tr = denorm_d(x_np[:, 1])

    fig2, axs = plt.subplots(1, len(DURATIONS), figsize=(19.5, 4.2),
                              sharey=True, constrained_layout=True)
    fig2.suptitle(
        f'2C Iteration {iteration+1}  |  P(crystalline) by duration slice  |  '
        f'Next = {v_n:.0f} V / {d_n:.1f} ms / ZrO2={c_n:.2f}',
        fontsize=_FSP, fontweight='bold')

    last_im = None
    for ci, d_val in enumerate(DURATIONS):
        vvc, cvc, grid_vc, shp = _make_vc_grid(d_val)
        probs, _ = _predict_np(model, grid_vc, samples=128)
        p_vc = probs[1].reshape(shp)

        ax = axs[ci]
        last_im = ax.contourf(vvc, cvc, p_vc,
                               levels=np.linspace(0, 1, 31),
                               cmap='bwr', vmin=0, vmax=1)
        _safe_contour(ax, vvc, cvc, p_vc,
                      levels=[0.5], colors='black', linewidths=1.6)

        ms = np.isclose(d_tr, d_val, atol=0.20)
        for lbl, col in [(1, CRYST_COL), (0, AMORPH_COL)]:
            m = ms & (y_np == lbl)
            if m.any():
                ax.scatter(v_tr[m], x_np[:, 2][ms & (y_np == lbl)], color=col,
                           s=_SC, edgecolors='white', linewidths=1.0, zorder=5)
        if np.isclose(d_n, d_val, atol=0.20):
            ax.scatter(v_n, c_n, color='gold', s=_SS, marker='*',
                       edgecolors='black', linewidths=1.2, zorder=6)

        ax.set_title(f'{d_val:.1f} ms', fontsize=_FSL)
        ax.set_xlim(V_fine[0], V_fine[-1])
        ax.set_ylim(0, 1)
        ax.set_xticks(VOLTAGES)
        ax.set_xlabel('Voltage (V)', fontsize=_FL)
        ax.tick_params(axis='x', rotation=45, labelsize=_FT)
        ax.tick_params(axis='y', labelsize=_FT)
        if ci == 0:
            ax.set_ylabel('ZrO2 fraction', fontsize=_FL)

    if last_im is not None:
        cb = fig2.colorbar(last_im, ax=axs.tolist(),
                           label='P(crystalline)', shrink=0.85, pad=0.01,
                           format='%.2f')
        cb.set_label('P(crystalline)', fontsize=_FL)
        cb.ax.tick_params(labelsize=_FT)

    _save_iter_fig(fig2, '2c', iteration, 'duration_slices')
    plt.show()


# ── 3C iteration plot ──────────────────────────────────────────────────────
# Saves 3 separate files: row0_phase_blend, row1_pcryst, row2_entropy
def plot_3c_iter(model_3c, model_2c, train_x_3c, train_y_3c,
                 train_x_2c, train_y_2c, next_idx, cryst_idx, iteration):
    pt  = test_x_all[next_idx]
    v_n = float(denorm_v(pt[0]))
    d_n = float(denorm_d(pt[1]))
    c_n = float(pt[2])

    x3c_np = _tnp(train_x_3c); y3c_np = _tnp(train_y_3c).astype(int)
    v3c = denorm_v(x3c_np[:, 0])
    d3c = denorm_d(x3c_np[:, 1])
    c3c = x3c_np[:, 2]

    TITLE = (f'3C Iteration {iteration+1}  |  '
             f'Next = {v_n:.0f} V / {d_n:.1f} ms / ZrO2={c_n:.2f}')
    ROW_SIZE = (19.5, 4.2)

    # ── Pre-compute all slice data ─────────────────────────────────────────
    slices = []
    for d_val in DURATIONS:
        vvc, cvc, grid_vc, shp = _make_vc_grid(d_val)
        probs2, _ = _predict_np(model_2c, grid_vc, samples=128)
        p_cryst = probs2[1].reshape(shp)
        probs3, _ = _predict_np(model_3c, grid_vc, samples=128)
        p_mono  = probs3[0].reshape(shp)
        p_ortho = probs3[1].reshape(shp)
        p_tetra = probs3[2].reshape(shp)
        rgb = (p_mono[..., None]  * _PC_RGB[0]
               + p_ortho[..., None] * _PC_RGB[1]
               + p_tetra[..., None] * _PC_RGB[2]) * p_cryst[..., None]
        rgb = np.clip(rgb ** 0.55, 0, 1)
        eps = 1e-10
        p_eff = np.stack([1.0 - p_cryst,
                          p_cryst * p_mono,
                          p_cryst * p_ortho,
                          p_cryst * p_tetra], axis=0)
        H = -(p_eff * np.log(p_eff + eps)).sum(axis=0)
        slices.append(dict(vvc=vvc, cvc=cvc, p_cryst=p_cryst,
                           rgb=rgb, H=H, d_val=d_val,
                           ms3=np.isclose(d3c, d_val, atol=0.20)))

    def _draw_ax(ax, ms3, d_val):
        for lbl, col in PHASE_COLS.items():
            m = ms3 & (y3c_np == lbl)
            if m.any():
                ax.scatter(v3c[m], c3c[m], color=col, s=_SC,
                           edgecolors='white', linewidths=1.0, zorder=5)
        if np.isclose(d_n, d_val, atol=0.20):
            ax.scatter(v_n, c_n, color='gold', s=_SS, marker='*',
                       edgecolors='black', linewidths=1.2, zorder=6)
        ax.set_xlim(V_fine[0], V_fine[-1]); ax.set_ylim(0, 1)
        ax.set_xticks(VOLTAGES)
        ax.set_xlabel('Voltage (V)', fontsize=_FL)
        ax.tick_params(axis='x', rotation=45, labelsize=_FT)
        ax.tick_params(axis='y', labelsize=_FT)

    # ── Row 0: Phase blend ─────────────────────────────────────────────────
    fig0, axs0 = plt.subplots(1, len(DURATIONS), figsize=ROW_SIZE,
                               sharey=True, constrained_layout=True)
    fig0.suptitle(TITLE, fontsize=_FSP, fontweight='bold')
    for ci, s in enumerate(slices):
        ax = axs0[ci]
        ax.imshow(s['rgb'], origin='lower', aspect='auto',
                  extent=[V_fine[0], V_fine[-1], C_fine[0], C_fine[-1]],
                  interpolation='bilinear')
        _safe_contour(ax, s['vvc'], s['cvc'], s['p_cryst'],
                      levels=[0.5], colors='white', linewidths=1.6, alpha=0.9)
        _draw_ax(ax, s['ms3'], s['d_val'])
        ax.set_title(f"{s['d_val']:.1f} ms", fontsize=_FSL)
        if ci == 0:
            ax.set_ylabel('ZrO2 fraction', fontsize=_FL)
    _save_iter_fig(fig0, '3c', iteration, 'row0_phase_blend')
    plt.show()

    # ── Row 1: P(crystalline) ──────────────────────────────────────────────
    fig1, axs1 = plt.subplots(1, len(DURATIONS), figsize=ROW_SIZE,
                               sharey=True, constrained_layout=True)
    fig1.suptitle(TITLE, fontsize=_FSP, fontweight='bold')
    last_im1 = None
    for ci, s in enumerate(slices):
        ax = axs1[ci]
        last_im1 = ax.contourf(s['vvc'], s['cvc'], s['p_cryst'],
                                levels=np.linspace(0, 1, 31),
                                cmap='bwr', vmin=0, vmax=1)
        _safe_contour(ax, s['vvc'], s['cvc'], s['p_cryst'],
                      levels=[0.5], colors='black', linewidths=1.6)
        _draw_ax(ax, s['ms3'], s['d_val'])
        ax.set_title(f"{s['d_val']:.1f} ms", fontsize=_FSL)
        if ci == 0:
            ax.set_ylabel('ZrO2 fraction', fontsize=_FL)
    if last_im1 is not None:
        cb1 = fig1.colorbar(last_im1, ax=axs1.tolist(),
                            shrink=0.85, pad=0.01, format='%.2f')
        cb1.set_label('P(crystalline)', fontsize=_FL)
        cb1.ax.tick_params(labelsize=_FT)
    _save_iter_fig(fig1, '3c', iteration, 'row1_pcryst')
    plt.show()

    # ── Row 2: 4-class entropy ─────────────────────────────────────────────
    fig2, axs2 = plt.subplots(1, len(DURATIONS), figsize=ROW_SIZE,
                               sharey=True, constrained_layout=True)
    fig2.suptitle(TITLE, fontsize=_FSP, fontweight='bold')
    last_im2 = None
    for ci, s in enumerate(slices):
        ax = axs2[ci]
        last_im2 = ax.contourf(s['vvc'], s['cvc'], s['H'],
                                levels=24, cmap='viridis')
        _safe_contour(ax, s['vvc'], s['cvc'], s['p_cryst'],
                      levels=[0.5], colors='white', linewidths=1.6, alpha=0.9)
        _draw_ax(ax, s['ms3'], s['d_val'])
        ax.set_title(f"{s['d_val']:.1f} ms", fontsize=_FSL)
        if ci == 0:
            ax.set_ylabel('ZrO2 fraction', fontsize=_FL)
    if last_im2 is not None:
        cb2 = fig2.colorbar(last_im2, ax=axs2.tolist(),
                            shrink=0.85, pad=0.01, format='%.2f')
        cb2.set_label('4-class entropy', fontsize=_FL)
        cb2.ax.tick_params(labelsize=_FT)
    _save_iter_fig(fig2, '3c', iteration, 'row2_entropy')
    plt.show()


## 2C Classification  (Amorphous vs Crystalline)

In [ ]:
# 4 corners × 1 random composition each = 4 initial points
np.random.seed(42)
CORNERS = [(506, 0.1), (506, 10.1), (716, 0.1), (716, 10.1)]
init_comps = np.random.choice(COMPOSITIONS, size=4, replace=False)

init_x, init_y = [], []
for (v, d), c in zip(CORNERS, init_comps):
    init_x.append([norm_v(v), norm_d(d), c])
    init_y.append(lookup_2C(v, d, c))

train_x_2c = torch.tensor(init_x, dtype=torch.float32)
train_y_2c = torch.tensor(init_y, dtype=torch.long)

# Make sure the initial 2C model has both classes; otherwise Dirichlet likelihood
# collapses to one output class and P(crystalline) cannot be plotted.
extra_init_2c = []
if len(torch.unique(train_y_2c)) < 2:
    missing_label = 1 - int(train_y_2c[0].item())
    found_extra = False
    for v_extra, d_extra in VD_GRID:
        for c_extra in COMPOSITIONS:
            y_extra = lookup_2C(v_extra, d_extra, c_extra)
            if y_extra == missing_label:
                x_extra = [norm_v(v_extra), norm_d(d_extra), float(c_extra)]
                train_x_2c = torch.cat([train_x_2c, torch.tensor([x_extra], dtype=torch.float32)], dim=0)
                train_y_2c = torch.cat([train_y_2c, torch.tensor([y_extra], dtype=torch.long)], dim=0)
                extra_init_2c.append((v_extra, d_extra, float(c_extra), y_extra))
                found_extra = True
                break
        if found_extra:
            break
    if not found_extra:
        raise RuntimeError('Could not find an opposite-class 2C seed point in the ground truth grid.')

# Reset accuracy history whenever the 2C run is re-initialized.
accuracy_history = []

print('Initial 2C training points:')
for (v,d), c, y in zip(CORNERS, init_comps, init_y):
    print(f'  {v}V / {d}ms / ZrO₂={c:.3f}  →  {"crystalline" if y else "amorphous"}')

for v, d, c, y in extra_init_2c:
    print(f'  added class-balance seed: {v}V / {d}ms / ZrO2={c:.3f}  ->  {"crystalline" if y else "amorphous"}')

# Track which indices in test_x_all have been sampled
sampled_2c = set()
for (v, d), c in zip(CORNERS, init_comps):
    # find the nearest index in test_x_all
    pt = torch.tensor([norm_v(v), norm_d(d), c], dtype=torch.float32)
    dist = (test_x_all - pt).norm(dim=1)
    sampled_2c.add(int(dist.argmin().item()))
for v, d, c, _ in extra_init_2c:
    pt = torch.tensor([norm_v(v), norm_d(d), c], dtype=torch.float32)
    dist = (test_x_all - pt).norm(dim=1)
    sampled_2c.add(int(dist.argmin().item()))


In [ ]:
# Hyperparameters
NU           = 0.5
TRAIN_ITER   = 100
LR           = 0.05
N_SAMPLES    = 256
LS_CON       = Interval(0.05, 1.0)
OS_CON       = Interval(0.10, 5.0)

# Initial ARD lengthscale (V: ~1 grid step = 0.20, D: 0.25, C: 0.20)
init_ls_2c = torch.tensor([[[0.20, 0.25, 0.20]], [[0.20, 0.25, 0.20]]], dtype=torch.float32)

prev_ent_2c = None
N_conv_2c   = 0
it = -1

# No iteration cap — stops at convergence (4 consecutive <10% entropy change)
# or when all 480 discrete points are exhausted.
while True:
    it += 1
    print(f'\n=== 2C  Iteration {it+1}  (train={len(train_x_2c)}, sampled={len(sampled_2c)}/480) ===')

    model_2c, lik_2c = run_sweeps(
        train_x_2c, train_y_2c, TRAIN_ITER,
        nu=NU, init_ls=init_ls_2c, ls_con=LS_CON, os_con=OS_CON, lr=LR)

    probs_2c, _, unc_all, ent = acquisition(model_2c, test_x_all, 'entropy', num_samples=N_SAMPLES)
    record_accuracy('2C', it + 1, model_2c, probs_2c=probs_2c)
    next_idx = select_next(unc_all, sampled_2c)

    pt = test_x_all[next_idx]
    v_s, d_s = snap_vd(pt[0], pt[1])
    c_s = float(pt[2])
    y_new = lookup_2C(v_s, d_s, c_s)
    print(f'  -> selected ({v_s}V, {d_s}ms, ZrO2={c_s:.2f})  label={"crystalline" if y_new else "amorphous"}')

    # Plot the model that selected the point; new label is appended after.
    plot_2c_iter(model_2c, train_x_2c, train_y_2c, next_idx, sampled_2c, it)

    sampled_2c.add(next_idx)
    train_x_2c = torch.cat([train_x_2c, pt.unsqueeze(0)], dim=0)
    train_y_2c = torch.cat([train_y_2c, torch.tensor([y_new])], dim=0)

    init_ls_2c = model_2c.covar_module.base_kernel.lengthscale.detach().clone()

    # Convergence check
    if prev_ent_2c is not None:
        diff = abs(ent - prev_ent_2c) / (abs(prev_ent_2c) + 1e-10)
        print(f'  Entropy diff: {diff:.4f}')
        N_conv_2c = N_conv_2c + 1 if diff < 0.1 else 0
        if N_conv_2c >= 4:
            print('2C converged.')
            break
    prev_ent_2c = ent

    if len(sampled_2c) >= len(test_x_all):
        print('All 480 discrete points sampled — stopping 2C.')
        break

print('\nRetraining final 2C model on all acquired 2C points...')
model_2c_final, lik_2c_final = run_sweeps(
    train_x_2c, train_y_2c, TRAIN_ITER,
    nu=NU, init_ls=init_ls_2c, ls_con=LS_CON, os_con=OS_CON, lr=LR, verbose=False)

train_x_2c_final = train_x_2c.clone()
train_y_2c_final = train_y_2c.clone()
model_2c = model_2c_final
lik_2c = lik_2c_final

print(f'Accuracy history saved to {ACCURACY_CSV}')
print(f'Accuracy plot saved to {ACCURACY_PNG}')
plot_accuracy_history(show=True)


## 2C → 3C Transition

In [ ]:
# ── 2C → 3C Transition  (principled: no bulk sweep) ──────────────────────
# Only (V, D, C) points already queried during 2C BO are used as 3C seed.
# Each point was a real measurement; composition is NOT re-swept for free.

init_3c_x, init_3c_y = [], []

for x, y in zip(train_x_2c_final, train_y_2c_final):
    if y.item() != 1:           # amorphous 2C points → skip
        continue
    v_s, d_s = snap_vd(x[0], x[1])
    c_s = float(x[2])
    lbl_3c = lookup_3C(v_s, d_s, c_s)
    if lbl_3c == -1:            # GT is amorphous despite 2C saying crystalline → skip
        continue
    init_3c_x.append([float(x[0]), float(x[1]), c_s])
    init_3c_y.append(lbl_3c)

# 2C training data carries over as-is — no expansion
train_x_2c = train_x_2c_final.clone()
train_y_2c = train_y_2c_final.clone()

train_x_3c = torch.tensor(init_3c_x, dtype=torch.float32)
train_y_3c = torch.tensor(init_3c_y, dtype=torch.long)

print(f'2C training data: {len(train_x_2c)} points')
print(f'  Amorphous={(train_y_2c==0).sum().item()}  Crystalline={(train_y_2c==1).sum().item()}')
print(f'\n3C seed data (2C crystalline queries only): {len(train_x_3c)} points')
for lbl, name in LABEL_NAME.items():
    n = (train_y_3c == lbl).sum().item() if len(train_y_3c) > 0 else 0
    print(f'  {name}: {n}')

n_present = len(train_y_3c.unique()) if len(train_y_3c) > 0 else 0
if n_present < 3:
    print(f'\nWARNING: only {n_present}/3 phase labels present in 3C seed.')
    print('3C GP may be unstable — consider running more 2C iterations first.')

print('\nRetraining 2C model on carried-over data...')
model_2c, lik_2c = run_sweeps(
    train_x_2c, train_y_2c, TRAIN_ITER,
    nu=NU, ls_con=LS_CON, os_con=OS_CON, lr=LR)


## 3C Classification  (Monoclinic / Orthorhombic / Tetragonal)

In [ ]:
TRAIN_ITER_3C = 100
ACQ_SCORE_MODE = '4class_entropy'
P_CRYST_POWER  = 1.0
LAMBDA_2C      = 0.0

init_ls_2c_3c = None
init_ls_3c    = None

# Exclude all 2C-queried points from 3C acquisition
sampled_3c = set()
for x in train_x_2c:
    sampled_3c.add(int((test_x_all - x).norm(dim=1).argmin().item()))
print(f'Initial 3C sampled/excluded points: {len(sampled_3c)}')

# Fixed 40-iteration run (no convergence stopping)
for it in range(50):
    print(f'\n=== 3C  Iteration {it+1}/50  (3C train={len(train_x_3c)}) ===')
    print(f'    Labels present: {sorted(train_y_3c.unique().tolist())}')

    # 1. Train 3C GP
    model_3c, lik_3c = run_sweeps(
        train_x_3c, train_y_3c, TRAIN_ITER_3C,
        nu=NU, init_ls=init_ls_3c, ls_con=LS_CON, os_con=OS_CON, lr=LR)
    ls_3c_before_update = model_3c.covar_module.base_kernel.lengthscale.detach().clone()

    # 2. 4-class acquisition over all 480 points (soft weighting, no hard cutoff)
    scores, ent_3c = acquisition_4class(
        model_2c, model_3c, test_x_all,
        lambda_2c=LAMBDA_2C, num_samples=N_SAMPLES,
        score_mode=ACQ_SCORE_MODE, p_cryst_power=P_CRYST_POWER)
    print(f'    acquisition={ACQ_SCORE_MODE}, p_cryst_power={P_CRYST_POWER}, lambda_2c={LAMBDA_2C}')
    print(f'    ent_conv (P_cryst * H_3c mean): {ent_3c:.4f}')

    scores_m = scores.clone()
    for i in sampled_3c:
        if i < len(scores_m):
            scores_m[i] = -float('inf')

    if torch.isneginf(scores_m).all():
        print('No unsampled points remain.')
        break

    best_global = int(scores_m.argmax().item())
    sampled_3c.add(best_global)

    pt  = test_x_all[best_global]
    v_s, d_s = snap_vd(pt[0], pt[1])
    c_s = float(pt[2])

    # cryst_idx for visualization reference only
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        f2_v = model_2c(test_x_all).sample(torch.Size([128]))
        p_cryst_vis = torch.softmax(f2_v, dim=1).mean(dim=0)[1]
    cryst_idx = (p_cryst_vis > 0.5).nonzero(as_tuple=True)[0]
    print(f'    Crystalline region (P>0.5): {len(cryst_idx)} / 480 points')

    # 3. Query ground truth
    lbl_3c = lookup_3C(v_s, d_s, c_s)
    lbl_2c = 0 if lbl_3c == -1 else 1
    print(f'  -> selected ({v_s}V, {d_s}ms, ZrO2={c_s:.2f})  '
          f'3C={LABEL_NAME.get(lbl_3c, "amorphous")}  2C={lbl_2c}')

    # 4. Update 3C training data (crystalline points only)
    if lbl_3c != -1:
        train_x_3c = torch.cat([train_x_3c, pt.unsqueeze(0)], dim=0)
        train_y_3c = torch.cat([train_y_3c, torch.tensor([lbl_3c])], dim=0)
        print('  Retraining 3C with new crystalline point...')
        model_3c, lik_3c = run_sweeps(
            train_x_3c, train_y_3c, TRAIN_ITER_3C,
            nu=NU, init_ls=ls_3c_before_update, ls_con=LS_CON, os_con=OS_CON, lr=LR)
    else:
        print('  Selected point is amorphous; 3C model unchanged.')

    # 5. Update 2C training data and retrain
    train_x_2c = torch.cat([train_x_2c, pt.unsqueeze(0)], dim=0)
    train_y_2c = torch.cat([train_y_2c, torch.tensor([lbl_2c])], dim=0)
    print('  Retraining 2C with new point...')
    model_2c, lik_2c = run_sweeps(
        train_x_2c, train_y_2c, TRAIN_ITER,
        nu=NU, init_ls=init_ls_2c_3c, ls_con=LS_CON, os_con=OS_CON, lr=LR)
    init_ls_2c_3c = model_2c.covar_module.base_kernel.lengthscale.detach().clone()

    # 6. Record accuracy after both models have been updated
    record_accuracy('3C', it + 1, model_2c, model_3c=model_3c)

    # 7. Visualize
    plot_3c_iter(model_3c, model_2c, train_x_3c, train_y_3c,
                 train_x_2c, train_y_2c, best_global, cryst_idx, it)

    init_ls_3c = model_3c.covar_module.base_kernel.lengthscale.detach().clone()

    if len(sampled_3c) >= len(test_x_all):
        print('All 480 discrete points sampled — stopping 3C.')
        break

print(f'Final accuracy history saved to {ACCURACY_CSV}')
print(f'Final accuracy plot saved to {ACCURACY_PNG}')
plot_accuracy_history(show=True)


In [ ]:
# ── Color Palette Test (standalone — no model needed) ─────────────────────
from matplotlib import patheffects as _pe

PALETTES_TEST = {
    'Current  (R-G-B)': np.array([[0.906, 0.298, 0.235],   # red
                                   [0.153, 0.682, 0.376],   # green
                                   [0.161, 0.502, 0.725]]), # blue
    'A  Sunset':         np.array([[0.910, 0.522, 0.239],   # burnt orange
                                   [0.169, 0.737, 0.694],   # teal
                                   [0.482, 0.435, 0.769]]), # lavender
    'B  Twilight':       np.array([[0.957, 0.518, 0.373],   # salmon
                                   [0.329, 0.627, 1.000],   # cornflower blue
                                   [0.373, 0.153, 0.804]]), # deep purple
    'C  Earth':          np.array([[0.831, 0.518, 0.353],   # terracotta
                                   [0.420, 0.659, 0.663],   # dusty teal
                                   [0.616, 0.494, 0.784]]), # lilac
    'D  Colorblind-Safe Scientific': np.array([[0.835, 0.369, 0.000],   # vermillion
                                                [0.000, 0.620, 0.451],   # bluish green
                                                [0.000, 0.447, 0.698]]), # blue

    'E  Mineral':                   np.array([[0.788, 0.435, 0.290],   # copper
                                                [0.263, 0.686, 0.647],   # sea glass
                                                [0.420, 0.388, 0.780]]), # soft indigo

    'F  Ink & Clay':                np.array([[0.722, 0.435, 0.322],   # clay
                                                [0.298, 0.604, 0.604],   # muted cyan
                                                [0.455, 0.408, 0.659]]), # slate violet
'G  Okabe-Ito (Scientific)':    np.array([[0.902, 0.624, 0.000],   # orange
                                              [0.337, 0.706, 0.914],   # sky blue
                                              [0.000, 0.620, 0.451]]), # bluish green

    'H  Cyberpunk Neon':            np.array([[1.000, 0.000, 0.741],   # neon magenta
                                              [0.000, 0.992, 0.992],   # cyan
                                              [1.000, 0.961, 0.000]]), # vivid yellow

    'I  High Contrast Classic':     np.array([[0.902, 0.098, 0.294],   # crimson red
                                              [0.235, 0.706, 0.294],   # emerald green
                                              [0.263, 0.388, 0.847]]), # royal blue

    'J  Vibrant UI':                np.array([[0.882, 0.094, 0.271],   # cranberry
                                              [0.529, 0.914, 0.067],   # fluorescent green
                                              [0.000, 0.341, 0.914]]), # moroccan blue

    'K  Pastel Contrast':           np.array([[1.000, 0.435, 0.380],   # coral pink
                                              [0.000, 0.800, 0.800],   # bright teal
                                              [0.400, 0.400, 1.000]]), # periwinkle blue
'L  Nature Publication':        np.array([[0.706, 0.016, 0.149],   # garnet red
                                              [0.016, 0.361, 0.733],   # cobalt blue
                                              [0.906, 0.671, 0.141]]), # goldenrod

    'M  Cinematic Teal & Orange':  np.array([[0.875, 0.400, 0.141],   # burnt orange
                                              [0.059, 0.404, 0.471],   # deep teal
                                              [0.914, 0.820, 0.608]]), # pale cream/gold

    'N  Modern Tech Data':          np.array([[0.902, 0.220, 0.498],   # vibrant magenta
                                              [0.118, 0.533, 0.898],   # azure blue
                                              [0.678, 0.847, 0.153]]), # yellow-green

    'O  Sophisticated Contrast':    np.array([[0.863, 0.667, 0.188],   # mustard
                                              [0.055, 0.443, 0.310],   # emerald
                                              [0.796, 0.337, 0.216]]), # terracotta

    'P  Dark Mode Pop':            np.array([[0.224, 0.902, 0.549],   # neon mint
                                              [0.667, 0.333, 0.961],   # electric purple
                                              [1.000, 0.400, 0.000]]), # vivid orange
    'MOT publication': np.array([
        [0.706, 0.016, 0.149],   # Monoclinic, garnet red
        [0.016, 0.361, 0.733],   # Orthorhombic, cobalt blue
        [0.906, 0.671, 0.141],   # Tetragonal, goldenrod
    ])
}
PHASE_NAMES_TEST = ['Monoclinic', 'Orthorhombic', 'Tetragonal']

# Synthetic 200×200 probability field
_N = 200
_x = np.linspace(0, 1, _N)
_y = np.linspace(0, 1, _N)
_XX, _YY = np.meshgrid(_x, _y)

def _sm3(a, b, c, T=7):
    e = np.exp(np.stack([a, b, c]) * T)
    return e / e.sum(axis=0)

# Phase regions: mono=left, ortho=center, tetra=right
_pm, _po, _pt = _sm3(1 - _XX, 1 - np.abs(_XX - 0.5) * 2, _XX)
# P(crystalline): bottom=amorphous (dark), top=crystalline (vivid)
_pc = np.clip(_YY * 1.4 - 0.1, 0, 1)

# Swatches: [label, p_mono, p_ortho, p_tetra, p_cryst]
_SWATCHES = [
    ('Monoclinic',         1.0, 0.0, 0.0, 1.0),
    ('Orthorhombic',       0.0, 1.0, 0.0, 1.0),
    ('Tetragonal',         0.0, 0.0, 1.0, 1.0),
    ('Mono + Ortho (50/50)', 0.5, 0.5, 0.0, 1.0),
    ('Ortho + Tetra (50/50)', 0.0, 0.5, 0.5, 1.0),
    ('Mono + Tetra (50/50)', 0.5, 0.0, 0.5, 1.0),
    ('Equal mix (uncertain)', 1/3, 1/3, 1/3, 1.0),
    ('Amorphous (P_cryst=0.1)', 1/3, 1/3, 1/3, 0.1),
]

fig, axs = plt.subplots(
    len(PALETTES_TEST), 2,
    figsize=(15, 4.2 * len(PALETTES_TEST)),
    gridspec_kw={'width_ratios': [2.8, 1]},
    constrained_layout=True,
)
fig.suptitle(
    'Color Palette Comparison\n'
    'Left: synthetic phase blend  |  Right: pure & mixed swatches\n'
    '(bottom of map = amorphous/dark,  top = crystalline/vivid,  white line = P(cryst)=0.5)',
    fontsize=12, fontweight='bold',
)

_stroke = [_pe.withStroke(linewidth=2.2, foreground='black')]

for ri, (pal_name, pc_arr) in enumerate(PALETTES_TEST.items()):

    # ── Blend map ─────────────────────────────────────────────────────────
    rgb_map = (_pm[..., None] * pc_arr[0]
               + _po[..., None] * pc_arr[1]
               + _pt[..., None] * pc_arr[2]) * _pc[..., None]
    rgb_map = np.clip(rgb_map ** 0.55, 0, 1)

    ax_m = axs[ri, 0]
    ax_m.imshow(rgb_map, origin='lower', aspect='auto',
                extent=[0, 1, 0, 1], interpolation='bilinear')
    ax_m.contour(_XX, _YY, _pc, levels=[0.5],
                 colors='white', linewidths=1.8, alpha=0.92)
    ax_m.set_title(pal_name, fontsize=11, fontweight='bold', loc='left', pad=6)
    ax_m.set_xlabel('← Mono                  Ortho                  Tetra →', fontsize=8)
    ax_m.set_ylabel('← Amorphous               Crystalline →', fontsize=8)
    ax_m.set_xticks([]); ax_m.set_yticks([])
    for tx, label in zip([0.1, 0.5, 0.9], PHASE_NAMES_TEST):
        ax_m.text(tx, 0.96, label, ha='center', va='top',
                  fontsize=8.5, fontweight='bold', color='white',
                  transform=ax_m.transAxes, path_effects=_stroke)

    # ── Swatches ──────────────────────────────────────────────────────────
    ax_s = axs[ri, 1]
    n_sw = len(_SWATCHES)
    ax_s.set_xlim(0, 1)
    ax_s.set_ylim(0, n_sw)
    ax_s.axis('off')

    for si, (sw_label, pm, po, pt, pcryst) in enumerate(_SWATCHES):
        raw = (pm * pc_arr[0] + po * pc_arr[1] + pt * pc_arr[2]) * pcryst
        col = np.clip(np.asarray(raw) ** 0.55, 0, 1)
        y = n_sw - 1 - si
        # color block
        ax_s.add_patch(plt.Rectangle(
            [0.01, y + 0.12], 0.30, 0.76,
            color=col, transform=ax_s.transData, clip_on=False,
        ))
        # black border so near-black swatches are visible
        ax_s.add_patch(plt.Rectangle(
            [0.01, y + 0.12], 0.30, 0.76,
            fill=False, edgecolor='#888888', linewidth=0.6,
            transform=ax_s.transData, clip_on=False,
        ))
        ax_s.text(0.37, y + 0.50, sw_label, va='center', fontsize=7.5,
                  transform=ax_s.transData)

    # separator line between palettes
    if ri < len(PALETTES_TEST) - 1:
        for ax in axs[ri]:
            ax.spines['bottom'].set_visible(True)
            ax.spines['bottom'].set_linewidth(0.5)

plt.show()
print('색 조합 고르셨으면 cell-viz의 _PC_RGB와 PHASE_COLS를 업데이트하면 됩니다.')
